In [ ]:
:set -XRankNTypes
:set -XScopedTypeVariables
putStrLn "Setup done."

# 🧮 Типы как алгебра

> Почему ADT называются *алгебраическими*? Потому что типы ведут себя вточно как числа —
> складываются, перемножаются, возводятся в степень. И из этого
> вырастает целый мир — от **Zipper**ов до коммутативных диаграмм.

**Предварительные знания:** базовый Haskell — `data`, `Maybe`, `Either`, паттерн-матчинг. [BaseHaskell.ipynb](BaseHaskell.ipynb)

**Содержание:**
1. Кардинальность типа: чем тип похож на число
2. Полукольцо типов: +, ×, степень
3. Изоморфизмы типов — алгебраические тождества в коде
4. Рекурсивные типы как уравнения
5. Производные типов — Zipper
6. Связь с теорией категорий


---
## 1️⃣ Кардинальность типа: каждый тип — это число

**Кардинальность** типа `T` — это количество его обитателей (значений). Обозначаем `|T|`.

| Тип | Обитатели | Кардинальность |
|------|----------|----------------|
| `Void` | никаких | 0 |
| `()` (единица) | `()` | 1 |
| `Bool` | `True`, `False` | 2 |
| `Maybe ()` | `Nothing`, `Just ()` | 2 |
| `Maybe Bool` | `Nothing`, `Just True`, `Just False` | 3 = 1 + 2 |
| `Either Bool Bool` | 4 варианта | 4 = 2 + 2 |
| `(Bool, Bool)` | 4 пары | 4 = 2 × 2 |
| `Bool -> Bool` | 4 функции | 4 = 2² |

> 💡 Уже видна закономерность: `Maybe a` содержит `1 + |a|` обитателей,
> `(a, b)` — `|a| × |b|`, функция `a -> b` — `|b|^|a|`.


In [1]:
import Data.Void (Void, absurd)

-- Void: 0 обитателей — нельзя создать значение этого типа!
-- voidValue :: Void  -- невозможно

-- (): ровно 1 обитатель
unit :: ()
unit = ()

-- Bool: 2 обитателя
bools :: [Bool]
bools = [minBound..maxBound]

-- Maybe Bool: 3 обитателя = 1 + 2
maybeBools :: [Maybe Bool]
maybeBools = [Nothing, Just False, Just True]

-- (Bool, Bool): 4 обитателя = 2 × 2
pairBools :: [(Bool, Bool)]
pairBools = [(a,b) | a <- [False,True], b <- [False,True]]

-- Bool -> Bool: 4 функции = 2^2
boolFuncs :: [Bool -> Bool]
boolFuncs = [const False, const True, id, not]

putStrLn $ "Maybe Bool: " ++ show (length maybeBools) ++ " = 1 + 2"
putStrLn $ "(Bool,Bool): " ++ show (length pairBools) ++ " = 2 x 2"
putStrLn $ "Bool->Bool: " ++ show (length boolFuncs) ++ " = 2^2"
putStrLn $ "absurd :: Void -> a (ex falso quodlibet)"

Line 29: Redundant $
Found:
putStrLn $ "absurd :: Void -> a (ex falso quodlibet)"
Why not:
putStrLn "absurd :: Void -> a (ex falso quodlibet)"

Maybe Bool: 3 = 1 + 2

(Bool,Bool): 4 = 2 x 2

Bool->Bool: 4 = 2^2

absurd :: Void -> a (ex falso quodlibet)

---
## 2️⃣ Полукольцо типов

Типы образуют **полукольцо** (по количеству обитателей).

| Алгебра | Типы | На числах |
|--------|-------|--------|
| Ноль | `Void` | 0 |
| Единица | `()` | 1 |
| Сложение | `Either a b` | |a| + |b| |
| Умножение | `(a, b)` | |a| × |b| |
| Степень | `a -> b` | |b|^|a| |

**Законы полукольца** (Up to isomorphism):

```
Either Void a  ≅  a               -- 0 + a = a
Either a Void  ≅  a               -- a + 0 = a
((), a)        ≅  a               -- 1 × a = a
(Void, a)      ≅  Void            -- 0 × a = 0
Either a b     ≅  Either b a      -- a + b = b + a
(a, b)         ≅  (b, a)          -- a × b = b × a
(a, Either b c) ≅ Either (a,b) (a,c)  -- дистрибутивность
```

> 🔥 **Почему не кольцо?** Потому что `Either a a` ≅ `(Bool, a)`, но обратное
> вычитание (`a - b`) для типов не определено канонически.
> (Есть расширения: дифференциальные категории — но это уже другая история.)


In [2]:
-- Изоморфизмы как функции Haskell
-- Either Void a ≅ a
fromVoidEither :: Either Void a -> a
fromVoidEither (Left v)  = absurd v
fromVoidEither (Right x) = x

toVoidEither :: a -> Either Void a
toVoidEither = Right

-- ((), a) ≅ a
fromUnitPair :: ((), a) -> a
fromUnitPair ((), x) = x

toUnitPair :: a -> ((), a)
toUnitPair x = ((), x)

-- Дистрибутивность: (a, Either b c) ≅ Either (a,b) (a,c)
distribute :: (a, Either b c) -> Either (a,b) (a,c)
distribute (x, Left y)  = Left  (x, y)
distribute (x, Right z) = Right (x, z)

factor :: Either (a,b) (a,c) -> (a, Either b c)
factor (Left  (x,y)) = (x, Left y)
factor (Right (x,z)) = (x, Right z)

-- Проверка
let ex1 = fromVoidEither (Right 42 :: Either Void Int)
let ex2 = distribute (True, Left 'a' :: Either Char Int)
print ex1  -- 42
print ex2  -- Left (True,'a')

42

Left (True,'a')

---
## 3️⃣ Изоморфизмы типов — алгебраические тождества в коде

Изоморфизм типов `A ≅ B` значит: есть взаимнообратные функции `f :: A -> B` и `g :: B -> A`,
такие что `g . f = id` и `f . g = id`.

Важнейшие изоморфизмы с алгебраическим смыслом:

```
(a^b)^c  ≅  a^(b×c)          -- каррирование!
a^(b+c)  ≅  a^b × a^c       -- функция из копроизведения
(a×b)^c  ≅  a^c × b^c       -- функция в произведение
a^1      ≅  a               -- единственная функция () -> a соответствует a
a^0      ≅  ()              -- функция Void -> a — ровно одна!
1^a      ≅  ()              -- функция a -> () единственна
```

**Звёздный момент:** каррирование — это `(b × c) -> a ≅ b -> (c -> a)`,
именно математическое равенство `a^(b×c) = (a^c)^b`. **`curry`/`uncurry` — это буквально алгебраическое тождество.**


In [3]:
-- Каррирование: a^(b×c) ≅ (a^c)^b
-- curry   :: ((b, c) -> a) -> b -> c -> a
-- uncurry :: (b -> c -> a) -> (b, c) -> a

-- Функция из суммы: a^(b+c) ≅ a^b × a^c
-- Either b c -> a  ≅  (b -> a, c -> a)
fromEitherFn :: (Either b c -> a) -> (b -> a, c -> a)
fromEitherFn f = (f . Left, f . Right)

toEitherFn :: (b -> a, c -> a) -> Either b c -> a
toEitherFn (f, g) (Left x)  = f x
toEitherFn (f, g) (Right y) = g y

-- Функция в произведение: (a×b)^c ≅ a^c × b^c
-- c -> (a, b)  ≅  (c -> a, c -> b)
fromPairFn :: (c -> (a, b)) -> (c -> a, c -> b)
fromPairFn f = (fst . f, snd . f)

toPairFn :: (c -> a, c -> b) -> c -> (a, b)
toPairFn (f, g) x = (f x, g x)

-- a^0 ≅ (): функция из Void уникальна
voidFn :: Void -> Int
voidFn = absurd

-- Демо
let add = uncurry (+) :: (Int, Int) -> Int
let addC = curry add  :: Int -> Int -> Int
putStrLn $ "curry/uncurry: " ++ show (add (3,4)) ++ " = " ++ show (addC 3 4)
let (showFn, negFn) = fromEitherFn (\e -> case e of Left n -> show n; Right b -> if b then "T" else "F")
putStrLn $ showFn 42 ++ " / " ++ negFn True

curry/uncurry: 7 = 7

42 / T

---
## 4️⃣ Рекурсивные типы как уравнения

Рекурсивный тип — это **решение алгебраического уравнения**.

**Список** `List a`:
```
L = 1 + a × L
```
Решаем как геометрическую серию (`a^0 + a^1 + a^2 + ...`):
```
L = 1/(1-a)    -- формально, как ряд Тейлора
```

**Дерево** `Tree a`:
```
T = 1 + a × T × T
```

**Числа Каталана** появляются как коэффициенты этого ряда!

> 🤯 Это не шутка: каждый коэффициент ряда типа дерева — это число Catalan,
> которое считает деревья с n внутренними узлами. [Joyal, 1981]


In [4]:
-- List a = 1 + a × List a
data MyList a = Nil | Cons a (MyList a) deriving (Show)

-- Tree a = 1 + a × Tree a × Tree a
data Tree a = Leaf | Node a (Tree a) (Tree a) deriving (Show)

-- Подсчёт размера как «вычисление» уравнения
listSize :: MyList a -> Int
listSize Nil         = 0   -- 1 -> 0 (размер, не кардинальность)
listSize (Cons _ xs) = 1 + listSize xs

treeSize :: Tree a -> Int
treeSize Leaf         = 0
treeSize (Node _ l r) = 1 + treeSize l + treeSize r

-- Числа Каталана: количество бинарных деревьев с n внутренними узлами
catalan :: Int -> Integer
catalan 0 = 1
catalan n = sum [catalan k * catalan (n-1-k) | k <- [0..n-1]]

let myList = Cons 1 (Cons 2 (Cons 3 Nil))
let myTree = Node 1 (Node 2 Leaf Leaf) (Node 3 Leaf (Node 4 Leaf Leaf))
putStrLn $ "List size: " ++ show (listSize myList)
putStrLn $ "Tree size: " ++ show (treeSize myTree)
putStrLn $ "Catalan: " ++ show (map catalan [0..7])
putStrLn "^ = деревья с 0,1,2,...,7 внутренними узлами"

List size: 3

Tree size: 4

Catalan: [1,1,2,5,14,42,132,429]

^ = деревья с 0,1,2,...,7 внутренними узлами

---
## 5️⃣ Производные типов: Zipper — как в матанализе

Конор МакБрайд (1997): **производная типа `T` — это `T` с дыркой**.

| Тип | Производная |
|------|----------|
| `a` | `1` (`a' = 1`) |
| `a + b` | `a' + b'` |
| `a × b` | `a' × b + a × b'` |
| `a^n` | `n × a^(n-1)` |

**Список** `L = 1 + a × L`:
```
dL/da = 0 + (1 × L + a × dL/da)
dL/da = L + a × dL/da
dL/da × (1 - a) = L
dL/da = L²           -- производная списка = пара списков!
```
**Zipper списка = ([левый контекст], [правый контекст])** — именно `L × L`!

**Дерево** `T = 1 + a × T²`:
```
dT/da = T² + a × 2T × dT/da
```
**Zipper дерева = (текущий узел, список контекстов)**

> 💡 Зиппер — это не просто трюк: это структура данных с одним выделенным контекстно-зависимым
> местом (фокусом), позволяющая эффективно ходить по структуре. И она получается
> **дифференцированием** типа — без всякой ручной работы!


In [5]:
-- Zipper для списка = два списка (контекст слева, фокус + правый хвост)
-- dL/da = L² — производная списка!
data ListZipper a = LZ [a] a [a] deriving (Show)

fromList :: [a] -> Maybe (ListZipper a)
fromList []     = Nothing
fromList (x:xs) = Just (LZ [] x xs)

zipLeft :: ListZipper a -> Maybe (ListZipper a)
zipLeft (LZ [] _ _)     = Nothing
zipLeft (LZ (l:ls) x rs) = Just (LZ ls l (x:rs))

zipRight :: ListZipper a -> Maybe (ListZipper a)
zipRight (LZ _ _ [])     = Nothing
zipRight (LZ ls x (r:rs)) = Just (LZ (x:ls) r rs)

modifyFocus :: (a -> a) -> ListZipper a -> ListZipper a
modifyFocus f (LZ ls x rs) = LZ ls (f x) rs

toList :: ListZipper a -> [a]
toList (LZ ls x rs) = reverse ls ++ [x] ++ rs

-- Демо
let Just z0 = fromList [1,2,3,4,5 :: Int]
let Just z1 = zipRight z0  -- фокус на 2
let Just z2 = zipRight z1  -- фокус на 3
let z3 = modifyFocus (*10) z2  -- меняем 3 -> 30
putStrLn $ "Zipper: " ++ show z3
putStrLn $ "Back to list: " ++ show (toList z3)

Zipper: LZ [2,1] 30 [4,5]

Back to list: [1,2,30,4,5]

In [6]:
-- Zipper для бинарного дерева
-- dT/da включает «хлебные крошки» (breadcrumbs) — контекст пути
data BTree a = BLeaf | BNode a (BTree a) (BTree a) deriving (Show)

-- Контекст: откуда мы пришли
data Crumb a = LeftCrumb a (BTree a)   -- пришли налево, запомнили корень и правое дерево
             | RightCrumb a (BTree a)  -- пришли направо, запомнили корень и левое дерево
             deriving (Show)

type Breadcrumbs a = [Crumb a]
type TreeZipper a  = (BTree a, Breadcrumbs a)

goLeft :: TreeZipper a -> Maybe (TreeZipper a)
goLeft (BNode x l r, bs) = Just (l, LeftCrumb x r : bs)
goLeft _                  = Nothing

goRight :: TreeZipper a -> Maybe (TreeZipper a)
goRight (BNode x l r, bs) = Just (r, RightCrumb x l : bs)
goRight _                  = Nothing

goUp :: TreeZipper a -> Maybe (TreeZipper a)
goUp (t, LeftCrumb x r  : bs) = Just (BNode x t r, bs)
goUp (t, RightCrumb x l : bs) = Just (BNode x l t, bs)
goUp (_, [])                   = Nothing

-- Дерево: (1 (2 . .) (3 . (4 . .)))
myBTree :: BTree Int
myBTree = BNode 1 (BNode 2 BLeaf BLeaf) (BNode 3 BLeaf (BNode 4 BLeaf BLeaf))

let z = (myBTree, [])
let Just z1 = goRight z
let Just z2 = goRight z1
let (focus, crumbs) = z2
putStrLn $ "Focus: " ++ show focus
putStrLn $ "Crumbs depth: " ++ show (length crumbs)
let Just z3 = goUp z2
let Just z4 = goUp z3
let (root, _) = z4
putStrLn $ "Back to root: " ++ show root

Focus: BNode 4 BLeaf BLeaf

Crumbs depth: 2

Back to root: BNode 1 (BNode 2 BLeaf BLeaf) (BNode 3 BLeaf (BNode 4 BLeaf BLeaf))

---
## 6️⃣ Категорная точка зрения

В терминах теории категорий в **Hask** (категория Haskell-типов):

| Алгебра | Теор. категорий | Haskell |
|--------|-----------|--------|
| Ноль | начальный объект | `Void` |
| Единица | терминальный объект | `()` |
| Сложение | копроизведение (coproduct) | `Either a b` |
| Умножение | произведение (product) | `(a, b)` |
| Степень | внутренний hom-объект (exponential) | `a -> b` |

**Hask является декартовой замкнутой категорией:**
это означает, что произведение и копроизведение существуют, правило exponentiation (curry-Howard),
и есть соответствие **Curry–Howard–Lambek**:

```
Типы  ↔  Логические предложения  ↔  Объекты категории
a × b  ↔  A ∧ B (конъюнкция)  ↔  произведение
a + b  ↔  A ∨ B (дизъюнкция)  ↔  копроизведение
a → b  ↔  A ⇒ B (импликация)  ↔  exponential
Void   ↔  ⊥ (ложь)            ↔  начальный объект
()     ↔  ⊤ (истина)          ↔  терминальный объект
```

> **Функция = доказательство импликации.** Написать `f :: a -> b` — это доказать `A ⇒ B`.


In [7]:
-- Curry-Howard: типы = логические формулы, программы = доказательства

-- A ∧ B => A  (проекция = элиминация конъюнкции)
andElimL :: (a, b) -> a
andElimL = fst

-- A => A ∧ A  (диагональ)
andIntro :: a -> (a, a)
andIntro x = (x, x)

-- A => A ∨ B  (введение дизъюнкции)
orIntroL :: a -> Either a b
orIntroL = Left

-- (A ∨ B), (A => C), (B => C) => C  (устранение дизъюнкции)
orElim :: Either a b -> (a -> c) -> (b -> c) -> c
orElim (Left x)  f _ = f x
orElim (Right y) _ g = g y

-- ⊥ => A  (ex falso quodlibet — из лжи следует что угодно)
exFalso :: Void -> a
exFalso = absurd

-- A ∧ (A => ⊥) => ⊥  (противоречие)
-- В Haskell нет ⊥ как типа, но можно показать через Double Negation
contradiction :: a -> (a -> Void) -> b
contradiction x f = absurd (f x)

putStrLn "Curry-Howard-Lambek:"
putStrLn "  программа :: (A, B) -> A  ==  доказательство A∧B => A"
putStrLn "  программа :: a -> (a, a)  ==  доказательство A => A∧A"
putStrLn "  absurd :: Void -> a       ==  ex falso quodlibet"
let result = orElim (Left 42 :: Either Int Bool) show (\b -> if b then "T" else "F")
putStrLn $ "orElim пример: " ++ result

Curry-Howard-Lambek:

  программа :: (A, B) -> A  ==  доказательство A∧B => A

  программа :: a -> (a, a)  ==  доказательство A => A∧A

  absurd :: Void -> a       ==  ex falso quodlibet

orElim пример: 42

---
## 🖼️ Диаграммы: полукольцо и категорные универсальные свойства

Ниже — SVG-диаграммы произведения и копроизведения в **Hask**:
- Произведение `(a, b)`: коммутативный квадрат с `fst`, `snd`
- Копроизведение `Either a b`: коммутативный квадрат с `Left`, `Right`
- Структура Zipperа: дерево с фокусом


![Произведение и копроизведение в Hask](../diagrams/algebras/ta_product_coproduct.svg)


![Полукольцо типов](../diagrams/algebras/ta_semiring.svg)


![Zipper списка](../diagrams/algebras/ta_zipper.svg)


---
## 🏁 Итог

| Концепция | Смысл |
|---------|------|
| `Void` = 0 | невозможное значение, `absurd` — единственная функция из `Void` |
| `()` = 1 | единственное значение |
| `Either` = + | кардинальность складывается |
| `(,)` = × | кардинальность перемножается |
| `->` = ^ | кардинальность возводится в степень |
| `curry` | мат.тождество `a^(b×c) = (a^c)^b` |
| `List a` = 1/(1-a) | числа Каталана — не шутка! |
| Zipper = d/da | производная типа = структура с фокусом |
| Curry–Howard–Lambek | тип = предложение логики = объект категории |

**Следующие шаги:**
- [Profunctors.ipynb](Profunctors.ipynb) — бивариантные функторы
- [Optics.ipynb](Optics.ipynb) — оптика через Zipper
- [MetaProgramming.ipynb](MetaProgramming.ipynb) — монада цитирования



---
**← Предыдущий:** [BaseHaskell](BaseHaskell.ipynb)  |  **Следующий →** [FunctorHierarchy](FunctorHierarchy.ipynb)